In [1]:
from sage.all import *
from tqdm import tqdm
from gmpy2 import gcdext
from math import comb


def sparse_ternary_sampler(n, k0, k1, fixed=False):
    if fixed:
        tmp_l = [1] * k0 + [-1] * k1 + [0] * (n-k0-k1)
        shuffle(tmp_l)
    else:
        tmp_l = []
        for _ in range(n):
            tmp_coin = randint(1, n)
            if tmp_coin <= k0:
                tmp_l.append(1)
            elif tmp_coin <= k0+k1:
                tmp_l.append(-1)
            else:
                tmp_l.append(0)
    return tmp_l


def sparse_vector_from_pdf(n, pdf, coin_range=2**32):
    tmp_pdf_list = []
    for vi in pdf:
        tmp_pdf_list.append([vi, sum([vi[1] for vi in tmp_pdf_list]) + int(coin_range*pdf[vi])])
    tmp_l = []
    for _ in range(n):
        tmp_coin = randint(0, coin_range-1)
        for i in range(len(tmp_pdf_list)):
            if tmp_coin < tmp_pdf_list[i][1]:
                tmp_l.append(tmp_pdf_list[i][0])
    return tmp_l


def ZZ_to_Fq(_vector, _q, _n):
    return [int(i % _q) if abs(_q - int(i % _q)) > int(i % _q) else (int(i % _q) - _q) for i in _vector] + [0] * (
            _n - len(_vector))

            
def get_conjugate(_f):
    _fl = list(_f) + [0] * (n - len(list(_f)))
    return PRQ([_fl[0]] + list(-vector(_fl[1:][::-1])))


def calculate_w(f, g, F, G, gamma):
    q00 = (gamma * f * PRz(get_conjugate(f)) + g * PRz(get_conjugate(g))) % mod_polynomial
    q01 = (gamma * F * PRz(get_conjugate(f)) + G * PRz(get_conjugate(g))) % mod_polynomial
    w = PRQ(q01) * PRQ(q00).inverse_mod(PRQ(mod_polynomial)) % PRQ(mod_polynomial)
    return w


def bound_round(tmp_w, bound=0.5):
    tmp_wz = []
    for vi in tmp_w:
        if RF(abs(vi) - floor(abs(vi))) >= bound:
            tmp_wz.append(vi.sign() * ceil(abs(vi)))
        else:
            tmp_wz.append(vi.sign() * floor(abs(vi)))
    return PRz(tmp_wz)


def Norm_polynomial(f, dims):
    fe, fo = (PRz([f[i * 2] for i in range(dims)]),
              PRz([f[i * 2 + 1] for i in range(dims)]))
    fx = fe(xz ** 2) - xz * fo(xz ** 2)
    Nf = (fe ** 2 - xz * fo ** 2) % (xz ** dims + 1)
    return Nf, fx


def Reduce_FG(f, g, F, G, dims):
    _mod = PRQ(xz ** dims + 1)
    _base = vector(PRQ, [f, g])
    _target = vector(PRQ, [F, G])
    _base_c = vector([get_conjugate(fi) for fi in _base])
    _q00 = _base * _base_c % _mod
    while True:
        _q01 = _target * _base_c % _mod
        _tmp = PRQ(_q01) * PRQ(_q00).inverse_mod(_mod) % _mod
        _k = PRQ(bound_round(_tmp))
        if _k == 0:
            return _target.change_ring(PRz)
        _target = (_target - _k * _base) % _mod


def calculate_FG(f, g, log_n):
    dim_list = [2 ** i for i in range(log_n, -1, -1)]
    f_prime, g_prime = f, g
    for j in range(1, log_n + 1):
        f_prime, _ = Norm_polynomial(f_prime, dim_list[j])
        g_prime, _ = Norm_polynomial(g_prime, dim_list[j])
    delta, u, v = gcdext(int(f_prime), int(g_prime))
    if delta != 1:
        raise ArithmeticError
    u = -u
    F = PRz(v * q)
    G = PRz(u * q)
    for i in range(log_n, 0, -1):
        f_prime, g_prime = f, g
        for j in range(1, i):
            f_prime, _ = Norm_polynomial(f_prime, dim_list[j])
            g_prime, _ = Norm_polynomial(g_prime, dim_list[j])
        dims = dim_list[i]
        N_f_prime, f_prime_x = Norm_polynomial(f_prime, dims)
        N_g_prime, g_prime_x = Norm_polynomial(g_prime, dims)
        next_mod = xz ** dim_list[i - 1] + 1
        F = (g_prime_x * F(xz ** 2)) % next_mod
        G = (f_prime_x * G(xz ** 2)) % next_mod
        F, G = Reduce_FG(f_prime, g_prime, F, G, dim_list[i - 1])
    return F, G

In [2]:
RF = RealField(prec=256)
PRz = PolynomialRing(ZZ, 'xz')
xz = PRz.gens()[0]
PRQ = PolynomialRing(QQ, 'xQ')

In [3]:
# END-512
n = 512
mod_polynomial = PRz.cyclotomic_polynomial(2 * n)
q = 257
PRq = PolynomialRing(Zmod(q), 'xq')
PRzm = PRz.quotient(mod_polynomial)
PRqm = PRq.quotient(mod_polynomial)
Q = 12289
kg = 72
kf = 72
de = 64
gamma = 4
e_pdf = {}
l0 = list(range(q))
l1 = [floor(floor(li * de / q) * q / de) - li + 3 for li in l0]
for l1i in l1:
    if l1i not in e_pdf:
        e_pdf[l1i] = l1.count(l1i) /  q

In [ ]:
test_num = 100
f_runs_length = 46
e_runs_length = 64
m0 = 3
chosen_continous_e_value = 2
print('log2(probability of bad secret key): %.1f' % RR(log(kf/n, 2) * f_runs_length))
print('log2(probability of bad randomness): %.1f, the opposite number of this is log2(precomputation cost)' % RR(log(e_pdf[chosen_continous_e_value], 2) * e_runs_length))
ones_polynomial = PRz([1] * n)
egf_wrap_errors = []
eGF_wrap_errors = []
gsfe_FCL_fail_times = 0
GQsFQe_FCL_fail_times = 0
for _ in tqdm(range(test_num)):
    while True:
        try:
            g = PRz(sparse_ternary_sampler(n, kg, kg))
            f = PRz([1] * f_runs_length + sparse_ternary_sampler(n, kf, kf)[:n-f_runs_length])
            h = PRz(ZZ_to_Fq(list((PRqm(f).inverse() * PRqm(g)).lift()), q, n))
            F, G = calculate_FG(f, g, int(log(n, 2)))
            w = calculate_w(f, g, F, G, gamma)
            wQ = PRz([round(vi * Q) for vi in w])
            FQ = (Q * F - wQ * f) % mod_polynomial
            GQ = (Q * G - wQ * g) % mod_polynomial
            break
        except ArithmeticError as err:
            continue
    s = PRz([randint(0, 1) for _ in range(n)])
    e = PRz([chosen_continous_e_value] * e_runs_length + sparse_vector_from_pdf(n-e_runs_length, e_pdf))
    # centralization to make the mean values of s and e to be close to 0, this is exactly how it is implemented in the submission.
    gsfe = (g * (2 * s - ones_polynomial) + f * (2 * e - ones_polynomial)) % mod_polynomial
    GQsFQe = (GQ * (2 * s - ones_polynomial) + FQ * (2 * e - ones_polynomial)) % mod_polynomial
    gsfe_l = list(gsfe) + [0] * (n-1-gsfe.degree())
    GQsFQe_l = list(GQsFQe) + [0] * (n-1-GQsFQe.degree())
    egf_wrap_errors.append(sum([1 if abs(vi) > q else 0 for vi in gsfe_l]))
    eGF_wrap_errors.append(sum([1 if abs(vi) > q*Q else 0 for vi in GQsFQe_l]))
    # FCL check
    reduced_gsfe = ZZ_to_Fq(gsfe_l, 2 * q, n)
    reduced_GQsFQe = ZZ_to_Fq(GQsFQe_l, 2 * q * Q, n)
    ranked_gsfe = sorted([[abs(abs(reduced_gsfe[i]) - q), i] for i in range(n)], key=lambda x: x[0])
    gsfe_FCL_candidate = [vi[1] for vi in ranked_gsfe[:m0]]
    ranked_GQsFQe = sorted([[abs(abs(reduced_GQsFQe[i]) - q * Q), i] for i in range(n)], key=lambda x: x[0])
    GQsFQe_FCL_candidate = [vi[1] for vi in ranked_GQsFQe[:m0]]
    for i in range(n):
        if abs(gsfe_l[i]) > q and i not in gsfe_FCL_candidate:
            gsfe_FCL_fail_times += 1
            break
    for i in range(n):
        if abs(GQsFQe_l[i]) > q*Q and i not in GQsFQe_FCL_candidate:
            GQsFQe_FCL_fail_times += 1
            break
for i in range(5):
    print('%d wrap errors in   gs+fe ratio: %d' % (i, RR(egf_wrap_errors.count(i) / test_num * 100)) + '%')
print('# ' * 32)
for i in range(5):
    print('%d wrap errors in GQs+FQe ratio: %d' % (i, RR(eGF_wrap_errors.count(i) / test_num * 100)) + '%')
print('# ' * 32)
print('FCL success probability on   gs+fe: %d' % RR((1 - gsfe_FCL_fail_times / test_num) * 100) + '%')
print('FCL success probability on GQs+FQe: %d' % RR((1 - GQsFQe_FCL_success_times / test_num) * 100) + '%')

log2(probability of bad secret key): -130.2
log2(probability of bad randomness): -128.4, the opposite number of this is log2(precomputation cost)


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:30<00:00,  1.11it/s]

0 wrap errors in   gs+fe ratio: 92%
1 wrap errors in   gs+fe ratio: 6%
2 wrap errors in   gs+fe ratio: 2%
3 wrap errors in   gs+fe ratio: 0%
4 wrap errors in   gs+fe ratio: 0%
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
0 wrap errors in GQs+FQe ratio: 100%
1 wrap errors in GQs+FQe ratio: 0%
2 wrap errors in GQs+FQe ratio: 0%
3 wrap errors in GQs+FQe ratio: 0%
4 wrap errors in GQs+FQe ratio: 0%
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
FCL success probability on   gs+fe: 100%
FCL success probability on GQs+FQe: 100%


In [35]:
# END-1024
n = 1024
mod_polynomial = PRz.cyclotomic_polynomial(2 * n)
q = 257
PRq = PolynomialRing(Zmod(q), 'xq')
PRzm = PRz.quotient(mod_polynomial)
PRqm = PRq.quotient(mod_polynomial)
kg = 96
kf = 96
de = 90
gamma = 3
e_pdf = {}
l0 = list(range(q))
l1 = [floor(floor(li * de / q) * q / de) - li + 2 for li in l0]
for l1i in l1:
    if l1i not in e_pdf:
        e_pdf[l1i] = l1.count(l1i) /  q

In [36]:
test_num = 100
f_runs_length = 46
e_runs_length = 170
m0 = 3
chosen_continous_e_value = 1
print('log2(probability of bad secret key): %.1f' % RR(log(kf/n, 2) * f_runs_length))
print('log2(probability of bad randomness): %.2f, the opposite number of this is log2(precomputation cost)' % RR(log(e_pdf[chosen_continous_e_value], 2) * e_runs_length))
ones_polynomial = PRz([1] * n)
egf_wrap_errors = []
eGF_wrap_errors = []
gsfe_FCL_fail_times = 0
GQsFQe_FCL_fail_times = 0
for _ in tqdm(range(test_num)):
    while True:
        try:
            g = PRz(sparse_ternary_sampler(n, kg, kg))
            f = PRz([1] * f_runs_length + sparse_ternary_sampler(n, kf, kf)[:n-f_runs_length])
            h = PRz(ZZ_to_Fq(list((PRqm(f).inverse() * PRqm(g)).lift()), q, n))
            F, G = calculate_FG(f, g, int(log(n, 2)))
            w = calculate_w(f, g, F, G, gamma)
            wQ = PRz([round(vi * Q) for vi in w])
            FQ = (Q * F - wQ * f) % mod_polynomial
            GQ = (Q * G - wQ * g) % mod_polynomial
            break
        except ArithmeticError as err:
            continue
    s = PRz([randint(0, 1) for _ in range(n)])
    e = PRz([chosen_continous_e_value] * e_runs_length + sparse_vector_from_pdf(n-e_runs_length, e_pdf))
    # centralization to make the mean values of s and e to be close to 0, this is exactly how it is implemented in the submission.
    gsfe = (g * (2 * s - ones_polynomial) + f * (2 * e - ones_polynomial)) % mod_polynomial
    GQsFQe = (GQ * (2 * s - ones_polynomial) + FQ * (2 * e - ones_polynomial)) % mod_polynomial
    gsfe_l = list(gsfe) + [0] * (n-1-gsfe.degree())
    GQsFQe_l = list(GQsFQe) + [0] * (n-1-GQsFQe.degree())
    egf_wrap_errors.append(sum([1 if abs(vi) > q else 0 for vi in gsfe_l]))
    eGF_wrap_errors.append(sum([1 if abs(vi) > q*Q else 0 for vi in GQsFQe_l]))
    # FCL check
    reduced_gsfe = ZZ_to_Fq(gsfe_l, 2 * q, n)
    reduced_GQsFQe = ZZ_to_Fq(GQsFQe_l, 2 * q * Q, n)
    ranked_gsfe = sorted([[abs(abs(reduced_gsfe[i]) - q), i] for i in range(n)], key=lambda x: x[0])
    gsfe_FCL_candidate = [vi[1] for vi in ranked_gsfe[:m0]]
    ranked_GQsFQe = sorted([[abs(abs(reduced_GQsFQe[i]) - q * Q), i] for i in range(n)], key=lambda x: x[0])
    GQsFQe_FCL_candidate = [vi[1] for vi in ranked_GQsFQe[:m0]]
    for i in range(n):
        if abs(gsfe_l[i]) > q and i not in gsfe_FCL_candidate:
            gsfe_FCL_fail_times += 1
        if abs(GQsFQe_l[i]) > q*Q and i not in GQsFQe_FCL_candidate:
            GQsFQe_FCL_fail_times += 1
for i in range(5):
    print('%d wrap errors in   gs+fe ratio: %d' % (i, RR(egf_wrap_errors.count(i) / test_num * 100)) + '%')
print('# ' * 32)
for i in range(5):
    print('%d wrap errors in GQs+FQe ratio: %d' % (i, RR(eGF_wrap_errors.count(i) / test_num * 100)) + '%')
print('# ' * 32)
print('FCL success probability on   gs+fe: %d' % RR((1 - gsfe_FCL_fail_times / test_num) * 100) + '%')
print('FCL success probability on GQs+FQe: %d' % RR((1 - GQsFQe_FCL_success_times / test_num) * 100) + '%')

log2(probability of bad secret key): -157.1
log2(probability of bad randomness): -257.34, the opposite number of this is log2(precomputation cost)


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [09:13<00:00,  5.54s/it]

0 wrap errors in   gs+fe ratio: 100%
1 wrap errors in   gs+fe ratio: 0%
2 wrap errors in   gs+fe ratio: 0%
3 wrap errors in   gs+fe ratio: 0%
4 wrap errors in   gs+fe ratio: 0%
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
0 wrap errors in GQs+FQe ratio: 100%
1 wrap errors in GQs+FQe ratio: 0%
2 wrap errors in GQs+FQe ratio: 0%
3 wrap errors in GQs+FQe ratio: 0%
4 wrap errors in GQs+FQe ratio: 0%
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
FCL success probability on   gs+fe: 100%
FCL success probability on GQs+FQe: 100%
